# Record & Replay Tutorial

This tutorial demonstrates PyTestLab's **Record & Replay** functionality, which enables you to:

- 🎯 Create reproducible measurement procedures
- 🛡️ Validate measurement script integrity
- 🔬 Run measurements offline without hardware
- 🧪 Implement regression testing for lab procedures

We'll walk through recording a simulated measurement session and then replaying it with validation.

## Overview

The replay system works in two phases:

### 1. Recording Phase
- Your script runs against **real instruments** (we use `simulate=True` here for demonstration)
- The `SessionRecordingBackend` captures every SCPI command and response
- Results are stored in a structured YAML session file

### 2. Replay Phase  
- Your script runs against the `ReplayBackend`
- Each command is validated against the recorded sequence
- Recorded responses are returned exactly as captured
- Any deviation triggers a `ReplayMismatchError`

## Setup

Let's start by importing the necessary modules and setting up our measurement environment.

In [ ]:
from importlib.resources import as_file, files
from pathlib import Path

import yaml

from pytestlab import AutoInstrument
from pytestlab.errors import InstrumentCommunicationError, ReplayMismatchError
from pytestlab.instruments.backends import SimBackend
from pytestlab.instruments.backends.replay_backend import ReplayBackend
from pytestlab.instruments.backends.session_recording_backend import SessionRecordingBackend

session_file = Path("tutorial_session.yaml").resolve()
profile_key = "keysight/EDU36311A"
instrument_key = "psu"

## Part 1: Recording a Session

First, let's create a simple measurement procedure that we want to record. This example uses a simulated power supply, but in practice you would use real hardware.

In [ ]:
def power_supply_characterization(psu):
    """Sweep a simulated supply and observe voltage and current."""
    print(f"Device: {psu.id()}")
    psu.set_current(1, 0.1)
    psu.output(1, True)
    measurements = []
    try:
        for voltage in [1.0, 2.0, 3.0, 4.0, 5.0]:
            psu.set_voltage(1, voltage)
            measured_v = float(psu.read_voltage(1))
            measured_i = float(psu.read_current(1))
            measurements.append(
                {"set_voltage": voltage, "measured_voltage": measured_v, "measured_current": measured_i}
            )
            print(f"Set {voltage:g} V; read {measured_v:g} V, {measured_i:g} A")
    finally:
        psu.output(1, False)
        psu.set_voltage(1, 0.0)
    return measurements

Now let's record this procedure. In a real scenario, you would use actual instrument addresses, but here we'll use simulation for demonstration:

In [ ]:
def record_measurement_session():
    fixture = files("pytestlab.profiles").joinpath("keysight", "EDU36311A.yaml")
    with as_file(fixture) as path:
        backend = SessionRecordingBackend(SimBackend(path), str(session_file), instrument_key)
        psu = AutoInstrument.from_preset(profile_key, backend_override=backend)
        try:
            results = power_supply_characterization(psu)
            backend.save_session(profile_key)
            print(f"Recorded to {session_file.name}")
            return results
        finally:
            psu.close()


recorded_results = record_measurement_session()

Let's examine the recorded session file to understand its structure:

In [ ]:
# Load and display the session file
with open("tutorial_session.yaml") as f:
    session_data = yaml.safe_load(f)

print("📄 Session File Structure:")
print(yaml.dump(session_data, default_flow_style=False, indent=2)[:1000] + "...")

# Count recorded commands
command_count = len(session_data["psu"]["log"])
print(f"\n📊 Total commands recorded: {command_count}")

# Show command types
command_types = {}
for entry in session_data["psu"]["log"]:
    cmd_type = entry["type"]
    command_types[cmd_type] = command_types.get(cmd_type, 0) + 1

print("📈 Command breakdown:")
for cmd_type, count in command_types.items():
    print(f"  {cmd_type}: {count}")

## Part 2: Replaying the Session

Now let's replay the exact same measurement procedure using the recorded session:

In [ ]:
def replay_measurement_session():
    backend = ReplayBackend(session_file, instrument_key)
    psu = AutoInstrument.from_preset(profile_key, backend_override=backend)
    try:
        return power_supply_characterization(psu)
    finally:
        psu.close()


replayed_results = replay_measurement_session()

Let's verify that the replayed results exactly match the recorded ones:

In [ ]:
assert len(recorded_results) == 5
assert replayed_results == recorded_results
print(f"All {len(recorded_results)} replay measurements exactly match the recording.")

## Part 3: Validation and Error Detection

Now let's see what happens when we try to deviate from the recorded sequence:

In [ ]:
def test_sequence_validation():
    backend = ReplayBackend(session_file, instrument_key)
    psu = AutoInstrument.from_preset(profile_key, backend_override=backend)
    try:
        psu.id()
        psu.set_current(1, 0.1)
        psu.output(1, True)
        # The recording expects 1.0 V as its first setpoint.
        psu.set_voltage(1, 2.5)
    except InstrumentCommunicationError as error:
        # The public driver preserves the backend mismatch as the cause.
        if not isinstance(error.__cause__, ReplayMismatchError):
            raise
        print(f"Replay rejected the changed command: {error.__cause__}")
    else:
        raise AssertionError("Replay accepted a command that differs from the recording")
    finally:
        psu.close()


test_sequence_validation()

## Part 4: CLI workflow

The notebook writes an example script and a bench locked to built-in simulation
with `simulation_only: true`. Run the commands below from the same directory.
No instrument address or physical hardware is required.

In [ ]:
script_content = '''
def main(bench):
    psu = bench.psu
    psu.set_current(1, 0.05)
    psu.output(1, True)
    try:
        psu.set_voltage(1, 3.3)
        voltage = float(psu.read_voltage(1))
        current = float(psu.read_current(1))
    finally:
        psu.output(1, False)
    print(f"Measured: {voltage:g} V, {current:g} A")
    return {"voltage_v": voltage, "current_a": current}
'''

Path("cli_measurement.py").write_text(script_content.lstrip(), encoding="utf-8")
print("Created CLI measurement script: cli_measurement.py")

In [ ]:
bench_config = """
bench_name: Tutorial Bench
simulation_only: true
simulate: true
instruments:
  psu:
    profile: keysight/EDU36311A
"""

Path("tutorial_bench.yaml").write_text(bench_config.lstrip(), encoding="utf-8")
print("Created explicitly simulated bench configuration: tutorial_bench.yaml")

In [ ]:
print("🖥️  CLI COMMAND EXAMPLES:")
print("=" * 25)
print()
print("1️⃣  Record a measurement session:")
print(
    "   ptl replay record cli_measurement.py --bench tutorial_bench.yaml --output cli_session.yaml"
)
print()
print("2️⃣  Replay the recorded session:")
print("   ptl replay run cli_measurement.py --session cli_session.yaml")
print()
print("3️⃣  View replay command help:")
print("   ptl replay --help")
print()
print("📂 Files created for CLI workflow:")
print("   - cli_measurement.py (measurement script)")
print("   - tutorial_bench.yaml (bench configuration)")
print("   - tutorial_session.yaml (recorded session)")

## Part 5: Best Practices

Here are some key recommendations for effective use of the replay system:

In [ ]:
print("💡 BEST PRACTICES")
print("=" * 17)
print()
print("🎯 Recording Phase:")
print("   • Label simulated recordings and hardware recordings separately")
print("   • Use descriptive session file names (e.g., 'psu_characterization_v2.1.yaml')")
print("   • Include version information in session metadata")
print("   • Test your recording immediately after creation")
print()
print("🔄 Replay Phase:")
print("   • Use identical instrument profiles during recording and replay")
print("   • Handle ReplayMismatchError gracefully in production code")
print("   • Enable debug logging to troubleshoot sequence issues")
print()
print("📁 File Management:")
print("   • Store session files in version control")
print("   • Organize sessions by measurement type or instrument")
print("   • Include README files documenting session purposes")
print()
print("🧪 Testing:")
print("   • Create regression tests using recorded sessions")
print("   • Validate session integrity periodically")
print("   • Use replay mode for CI/CD pipeline testing")

## Summary

In this tutorial, we've explored PyTestLab's Record & Replay functionality:

✅ **Recording**: Captured real instrument interactions using `SessionRecordingBackend`  
✅ **Replaying**: Validated exact sequence reproduction using `ReplayBackend`  
✅ **Validation**: Demonstrated automatic detection of sequence deviations  
✅ **CLI Integration**: Showed command-line workflow for production use  
✅ **Best Practices**: Covered recommendations for effective replay system usage  

### Key Benefits Achieved:

- **🎯 Reproducibility**: Exact measurement sequences can be repeated indefinitely
- **🛡️ Integrity**: Scripts cannot accidentally deviate from validated procedures  
- **🔬 Offline Analysis**: Complex measurements run without physical instruments
- **🧪 Regression Testing**: Automated validation of measurement procedure consistency

### Next Steps:

1. Try recording your own measurement procedures with real instruments
2. Integrate replay validation into your testing workflows  
3. Explore multi-instrument session recording for complex setups
4. Set up automated regression testing using recorded sessions

The Record & Replay system provides a powerful foundation for reproducible, validated measurement automation in PyTestLab! 🚀